# NISAR atlas · Ostrava–Karviná: mining subsidence

**What it does.** Finds every 12-day NISAR GUNW interferogram (PROVISIONAL, from 17 Jun 2026) over the Ostrava–Karviná (CZ)
and Jastrzębie (PL) coal district. It reads only the study box from each granule over HTTPS (no gigabyte downloads),
computes the line-of-sight (LOS) velocity for each track, flags anomalous patches with an Isolation Forest, combines an
ascending and a descending track into vertical and east–west motion, and outlines subsidence bowls.

**Outputs** (in `output/ostrava_karvina/` at the repo root, ignored by git):
- `velocity_<track>.tif` — LOS velocity, cm/yr; `anomaly_<track>.tif` — Isolation Forest score (0..1)
- `zones_<track>.geojson` / `.csv` — anomalous sinking zones per track
- `stack_<track>.nc` — the cropped stack (tens of MB)
- `vertical_cm_yr.tif`, `east_cm_yr.tif`, `zones_vertical_clustered.geojson` / `.csv` — the bowls
- `osm_water.geojson`, `osm_heaps.geojson` — OSM water and spoil heaps used as masks
- PNG maps and time series

**How to run** (from the repo root):
1. A free NASA Earthdata account. Put your token in `~/.edl_token` or the `EARTHDATA_TOKEN` environment variable
   (never in the notebook or the repo). `~/.netrc` also works.
2. `uv run --with jupyterlab --with earthaccess --with xarray --with rioxarray --with netcdf4 --with geopandas --with osmnx --with scikit-image --with scikit-learn --with pandas --with pyproj jupyter lab scripts/ostrava/ostrava_karvina.ipynb`
3. Run all. Memory under 2 GB. Parallel reading uses `fork` (Linux, macOS); on Windows it falls back to one process.

Then run the Node scripts in `scripts/ostrava/` (see `README.md` there) for the OSM context and the results page.

## 1. Settings

In [ ]:
from pathlib import Path

# Study box: Ostrava – Karviná (CZ) + Jastrzębie-Zdrój (PL). lon/lat, WGS84
BBOX = (18.15, 49.70, 18.80, 50.05)          # (W, S, E, N)
DATE_START, DATE_END = "2026-06-17", "2026-11-14"
SHORT_NAME = "NISAR_L2_GUNW_PROVISIONAL_V1"

COH_MIN = 0.30        # below this a pixel counts as noise
MIN_VALID_FRAC = 0.6  # share of valid pairs a pixel needs
MIN_COVERAGE = 0.30   # skip a frame that covers < 30 % of the box (e.g. D080)
EXCLUDE_KEYS = []     # frames to drop by hand, e.g. ["D080_F063"]
EDGE_ERODE_PX = 4     # trim N pixels from the swath edge (edge artefacts)
DERAMP = True         # subtract a plane from each pair (residual atmosphere / ionosphere)
DECOMP_PAIR = None    # track pair for the vertical, e.g. ("A057_F027", "D152_F063"); None = automatic
LOOK_SIDE = "left"    # NISAR looks left (unlike Sentinel-1)
VERT_ZONE_CM = -15    # vertical-velocity threshold for a "bowl", cm/yr
HP_SIGMA_KM = 5       # long-wavelength noise filter (atmosphere): removes smooth patches wider than ~5 km; 0 = off
HP_MASK_CM = 8        # pixels faster than this (|v|, cm/yr) are left out of the noise estimate (so bowls are not eaten)
CLUSTER_GAP_M = 400   # fragments closer than this merge into one bowl
MAX_WATER_FRAC = 0.30 # a bowl with more than this share of (OSM) water is dropped as an artefact
SPLIT_DEPTH_CM = 10   # a bowl with several centres is split if the "pass" between them is >= this (cm/yr)
SPLIT_SMOOTH_PX = 2   # smoothing before the centre search (80 m pixels)
WATER_FILE = None     # your own water GeoJSON instead of OSM (optional)
HEAP_FILE = None      # your own spoil-heap GeoJSON instead of OSM (optional)
MAX_HOLE_KM2 = 3      # a no-data "white centre" smaller than this and touching subsidence counts as part of the bowl
LOS_CONFIRM_CM = -8   # a track confirms a bowl if its mean LOS velocity there is faster than this
MIN_CONFIRM_TRACKS = 2  # how many LOS tracks must confirm a bowl
SIGN = -1             # phase -> LOS sign. Mining bowls must come out NEGATIVE (away from the satellite / down).
                      # If they are red / positive on the map, set SIGN = +1 and rerun.
USE_IONO = True       # subtract the ionosphere screen if the file has one
REF_LONLAT = None     # (lon, lat) of a stable point for the reference; None = scene median
REF_RADIUS_PX = 5
CONTAMINATION = 0.03  # share of pixels the Isolation Forest treats as anomalous
MIN_ZONE_PX = 20      # minimum zone size (20 px ≈ 0.13 km²)
MAX_FILES_PER_TRACK = 40
N_WORKERS = 8         # parallel reader processes (I/O-bound, so more than the cores is fine)
BLOCK_MB = 4          # HTTP cache block size; fewer requests = faster

# If the .h5 files are already on disk: a list of paths, and the search is skipped
LOCAL_FILES = None

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())   # the repo root
OUT = ROOT / "output" / "ostrava_karvina"
OUT.mkdir(parents=True, exist_ok=True)
print("Outputs ->", OUT)

## 2. Search and open the files (only the pieces we need, no full download)

In [ ]:
import os, re
from collections import defaultdict


def earthdata_login():
    """Log in from the environment: EARTHDATA_TOKEN, else ~/.edl_token, else ~/.netrc. Nothing is printed."""
    import earthaccess
    tok = Path.home() / ".edl_token"
    if not os.environ.get("EARTHDATA_TOKEN") and tok.exists():
        os.environ["EARTHDATA_TOKEN"] = tok.read_text().strip()
    try:
        earthaccess.login(strategy="environment")
    except Exception:
        earthaccess.login(strategy="netrc")
    return earthaccess


def search_and_open():
    earthaccess = earthdata_login()
    res = earthaccess.search_data(short_name=SHORT_NAME, bounding_box=BBOX,
                                  temporal=(DATE_START, DATE_END), count=500)
    print(f"Granules found: {len(res)}")
    names = [r["umm"]["GranuleUR"] for r in res]
    return names, res, earthaccess

if LOCAL_FILES:
    names = [Path(p).name for p in LOCAL_FILES]
    handles = list(LOCAL_FILES)
    print(f"Local files: {len(names)}")
else:
    names, granules, earthaccess = search_and_open()
    handles = None   # opened later, per track
for n in names[:5]: print("  ", n)

## 3. Reading GUNW: find the HDF5 paths once, then reuse them

In [ ]:
import h5py, numpy as np
from datetime import datetime
from pyproj import Transformer
from scipy import ndimage
from scipy.interpolate import RegularGridInterpolator

C = 299_792_458.0
_LAYOUT = {}   # cache of the paths inside the HDF5 (the same for all GUNW files)

def _datasets(h5):
    out = []
    h5.visititems(lambda n, o: out.append(n) if isinstance(o, h5py.Dataset) else None)
    return out

def _pick(paths, name, under=""):
    exact = [p for p in paths if p == f"{under}/{name}".lstrip("/")]
    if exact: return exact[0]
    c = [p for p in paths if p.startswith(under) and p.endswith("/" + name)]
    return min(c, key=len) if c else None

def _str(v):
    v = v[()] if hasattr(v, "shape") else v
    if isinstance(v, np.ndarray): v = v.ravel()[0]
    return v.decode() if isinstance(v, bytes) else str(v)

def _date(s):
    m = re.search(r"(\d{4})-?(\d{2})-?(\d{2})", s)
    return datetime(int(m[1]), int(m[2]), int(m[3]))

def _discover(h5):
    # full tree walk: expensive over HTTP, so done once
    P = _datasets(h5)
    unw = [p for p in P if p.endswith("/unwrappedPhase") and "unwrappedInterferogram" in p]
    if not unw:
        raise KeyError("unwrappedPhase not found. First paths:\n" + "\n".join(P[:40]))
    unw = unw[0]
    pol = unw.rsplit("/", 1)[0]; ifg = pol.rsplit("/", 1)[0]
    L = dict(unw=unw, pol=pol.rsplit("/", 1)[1],
             coh=_pick(P, "coherenceMagnitude", pol), cc=_pick(P, "connectedComponents", pol),
             iono=_pick(P, "ionospherePhaseScreen", pol),
             x=_pick(P, "xCoordinates", ifg), y=_pick(P, "yCoordinates", ifg),
             proj=_pick(P, "projection", ifg),
             freq=_pick(P, "centerFrequency", ifg.rsplit("/", 1)[0]),
             rt=_pick(P, "referenceZeroDopplerStartTime"), st=_pick(P, "secondaryZeroDopplerStartTime"),
             pdp=_pick(P, "orbitPassDirection"), tn=_pick(P, "trackNumber"))
    for nm in ("losUnitVectorX", "losUnitVectorY", "incidenceAngle"):
        q = _pick(P, nm); L[nm] = q
        if q:
            g = q.rsplit("/", 1)[0]
            L["cube_x"], L["cube_y"] = _pick(P, "xCoordinates", g), _pick(P, "yCoordinates", g)
            L["cube_h"] = _pick(P, "heightAboveEllipsoid", g)
    return L

def layout(h5):
    L = _LAYOUT.get("gunw")
    if L is None or L["unw"] not in h5:      # other polarization or structure -> search again
        L = _discover(h5); _LAYOUT["gunw"] = L
    return L

def describe(h5):
    L = layout(h5)
    d = {k: L[k] for k in ("unw", "pol", "coh", "cc", "iono", "x", "y")}
    pr = h5[L["proj"]]
    epsg = pr.attrs.get("epsg_code", None)
    d["epsg"] = int(np.ravel(epsg)[0]) if epsg is not None else int(pr[()])
    d["wavelength"] = C / float(np.ravel(h5[L["freq"]][()])[0]) if L["freq"] else 0.2384
    if L["rt"] and L["st"]:
        d["ref"], d["sec"] = _date(_str(h5[L["rt"]])), _date(_str(h5[L["st"]]))
    else:
        ds = sorted({datetime.strptime(t, "%Y%m%d") for t in re.findall(r"(20\d{6})T", str(h5.filename))})
        d["ref"], d["sec"] = ds[0], ds[-1]
    d["dir"] = _str(h5[L["pdp"]])[0].upper() if L["pdp"] else "?"
    d["track"] = int(np.ravel(h5[L["tn"]][()])[0]) if L["tn"] else -1
    return d

def _cube2d(h5, L, name, x, y, h_target=300.0):
    # 2-D slice of a metadata cube (height × y × x) on the crop grid
    p = L.get(name)
    if not p or not L.get("cube_x"): return None
    cx, cy = h5[L["cube_x"]][()], h5[L["cube_y"]][()]
    cube = h5[p]
    if cube.ndim == 3:
        k = int(np.argmin(abs(h5[L["cube_h"]][()] - h_target))) if L.get("cube_h") else cube.shape[0] // 2
        a = cube[k]
    else:
        a = cube[()]
    a = np.asarray(a, "f8")
    if cy[0] > cy[-1]: cy, a = cy[::-1], a[::-1]
    if cx[0] > cx[-1]: cx, a = cx[::-1], a[:, ::-1]
    f = RegularGridInterpolator((cy, cx), a, bounds_error=False, fill_value=None)
    YY, XX = np.meshgrid(y, x, indexing="ij")
    return f(np.c_[YY.ravel(), XX.ravel()]).reshape(len(y), len(x)).astype("f4")

def geometry(h5, x, y, direction):
    # unit vector target -> satellite (east, north, up) for each pixel
    L = layout(h5)
    ux, uy = _cube2d(h5, L, "losUnitVectorX", x, y), _cube2d(h5, L, "losUnitVectorY", x, y)
    if ux is not None and uy is not None:
        src = "losUnitVector"
        # target -> satellite. NISAR looks LEFT: on an ascending pass the satellite is east of the target
        # (east component > 0), on a descending pass west (< 0). For a right-looking radar it is the other way round.
        exp_pos = (direction == "A") == (LOOK_SIDE == "left")
        if (exp_pos and np.nanmean(ux) < 0) or (not exp_pos and np.nanmean(ux) > 0):
            ux, uy = -ux, -uy; src += " (flipped)"
        uz = np.sqrt(np.clip(1 - ux**2 - uy**2, 0, 1))
    else:
        inc = _cube2d(h5, L, "incidenceAngle", x, y)
        src = "incidenceAngle" if inc is not None else "const 39°"
        inc = np.deg2rad(inc if inc is not None else np.full((len(y), len(x)), 39.0, "f4"))
        a = np.deg2rad((348.0 if direction == "A" else 192.0) + (-90 if LOOK_SIDE == "left" else 90))
        ux, uy, uz = -np.sin(inc) * np.sin(a), -np.sin(inc) * np.cos(a), np.cos(inc)
    return ux.astype("f4"), uy.astype("f4"), uz.astype("f4"), src

def read_crop(h5, bbox=BBOX, want_geom=False):
    d = describe(h5)
    x, y = h5[d["x"]][()], h5[d["y"]][()]
    tr = Transformer.from_crs(4326, d["epsg"], always_xy=True)
    W, S, E, N = bbox
    xs, ys = tr.transform([W, E, W, E], [S, S, N, N])
    ix = np.where((x >= min(xs)) & (x <= max(xs)))[0]
    iy = np.where((y >= min(ys)) & (y <= max(ys)))[0]
    if len(ix) < 2 or len(iy) < 2:
        return None
    sl = (slice(iy[0], iy[-1] + 1), slice(ix[0], ix[-1] + 1))
    unw = h5[d["unw"]][sl].astype("float32")
    coh = h5[d["coh"]][sl].astype("float32") if d["coh"] else np.ones_like(unw)
    cc = h5[d["cc"]][sl] if d["cc"] else np.ones(unw.shape, "uint8")
    swath = np.isfinite(unw) & (unw != 0)
    if EDGE_ERODE_PX:
        swath = ndimage.binary_erosion(swath, iterations=EDGE_ERODE_PX, border_value=1)
    if USE_IONO and d["iono"]:
        iono = h5[d["iono"]][sl].astype("float32")
        if iono.shape == unw.shape:
            unw = unw - np.nan_to_num(iono)
    bad = ~swath | (coh < COH_MIN) | (cc == 0) | ~np.isfinite(unw)
    unw[bad] = np.nan
    disp = SIGN * unw * d["wavelength"] / (4 * np.pi)            # metres LOS
    out = dict(disp=disp, coh=coh, x=x[sl[1]], y=y[sl[0]], **{k: d[k] for k in ("epsg", "ref", "sec", "dir", "track", "pol", "wavelength")})
    if want_geom:
        out["ux"], out["uy"], out["uz"], out["geom_src"] = geometry(h5, out["x"], out["y"], d["dir"])
    return out
print("ready")

## 4. Crop all pairs in parallel and group them by frame

In [ ]:
import time, multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor, as_completed

def track_key_from_name(n):
    m = re.search(r"GUNW_\d{3}_(\d{3})_([AD])_(\d{3})", n)
    return f"{m[2]}{m[1]}_F{m[3]}" if m else "unknown"

groups = defaultdict(list)
for i, n in enumerate(names):
    groups[track_key_from_name(n)].append(i)
print({k: len(v) for k, v in groups.items()})

# tasks: (index, frame key, compute geometry?). Geometry is needed only for the first file of a frame
tasks = []
for key, idx in groups.items():
    if key in EXCLUDE_KEYS:
        print("  dropped by hand:", key); continue
    for j, i in enumerate(idx[:MAX_FILES_PER_TRACK]):
        tasks.append((i, key, j == 0))

# source for each file: a local path or an HTTPS URL
if handles:
    sources = {i: handles[i] for i, _, _ in tasks}
else:
    sources = {}
    for i, _, _ in tasks:
        links = granules[i].data_links(access="external")
        sources[i] = next((u for u in links if u.endswith(".h5")), links[0])

_FS = None
def _worker_init():
    global _FS
    if not handles:
        _FS = earthdata_login().get_fsspec_https_session()

def _job(t):
    i, key, want = t
    src = sources[i]
    try:
        f = src if handles else _FS.open(src, "rb", block_size=BLOCK_MB * 2**20, cache_type="blockcache")
        with h5py.File(f, "r") as h5:
            c = read_crop(h5, want_geom=want)
        return i, key, c, None
    except Exception as e:
        return i, key, None, repr(e)

t0 = time.time()
out = []
if N_WORKERS > 1 and "fork" in mp.get_all_start_methods():   # workers inherit the notebook state through fork
    ctx = mp.get_context("fork")
    with ProcessPoolExecutor(N_WORKERS, mp_context=ctx, initializer=_worker_init) as ex:
        futs = [ex.submit(_job, t) for t in tasks]
        for n, fu in enumerate(as_completed(futs), 1):
            out.append(fu.result())
            if n % 10 == 0 or n == len(futs): print(f"  {n}/{len(futs)}  {time.time()-t0:.0f} s")
else:
    _worker_init()
    for n, t in enumerate(tasks, 1):
        out.append(_job(t))
        if n % 10 == 0 or n == len(tasks): print(f"  {n}/{len(tasks)}  {time.time()-t0:.0f} s")

crops = defaultdict(list)
for i, key, c, err in sorted(out, key=lambda r: r[0]):
    if err:
        print("  ERROR", names[i][:70], "->", err); continue
    if c is None:
        print("  outside the box:", names[i][:70]); continue
    k = key if key != "unknown" else (f"{c['dir']}{c['track']:03d}" if c["track"] >= 0 else key)
    c["name"] = names[i]
    crops[k].append(c)
for k in sorted(crops):
    cl = sorted(crops[k], key=lambda c: c["ref"])
    g = next((c["geom_src"] for c in cl if "geom_src" in c), "—")
    cov = [np.isfinite(c["disp"]).mean() for c in cl]
    print(f"  {k}: {len(cl)} pairs, {cl[0]['ref']:%Y-%m-%d}…{cl[-1]['sec']:%Y-%m-%d}, "
          f"valid {min(cov):.0%}–{max(cov):.0%}, geometry: {g}")
for k in list(crops):
    cov = np.median([np.isfinite(c["disp"]).mean() for c in crops[k]])
    if cov < MIN_COVERAGE:
        print(f"  skipping {k}: median box coverage {cov:.0%} < {MIN_COVERAGE:.0%}")
        del crops[k]
print({k: len(v) for k, v in crops.items()}, f"· total {time.time()-t0:.0f} s")

## 5. Stack, reference and velocity for each track

In [ ]:
import xarray as xr, warnings
warnings.filterwarnings("ignore", "Mean of empty slice")
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", "All-NaN slice")

def build_stack(cl):
    # duplicates (the same pair, reprocessed): keep the last by name (newer CRID)
    uniq = {}
    for c in sorted(cl, key=lambda c: c.get("name", "")):
        uniq[(c["ref"], c["sec"])] = c
    if len(uniq) < len(cl):
        print(f"  duplicates removed: {len(cl) - len(uniq)}")
    cl = sorted(uniq.values(), key=lambda c: c["ref"])
    # common grid (intersection of coordinates)
    xs = cl[0]["x"]; ys = cl[0]["y"]
    for c in cl[1:]:
        xs = np.intersect1d(np.round(xs, 1), np.round(c["x"], 1))
        ys = np.intersect1d(np.round(ys, 1), np.round(c["y"], 1))
    if len(xs) < 2 or len(ys) < 2:
        raise ValueError("the pairs share no grid (different frames?)")
    ys = ys[::-1] if cl[0]["y"][0] > cl[0]["y"][-1] else ys
    D, K, G = [], [], None
    for c in cl:
        jx = np.searchsorted(np.round(c["x"], 1), xs)
        yy = np.round(c["y"], 1)
        jy = np.array([np.where(yy == v)[0][0] for v in ys])
        D.append(c["disp"][np.ix_(jy, jx)]); K.append(c["coh"][np.ix_(jy, jx)])
        if G is None and "ux" in c:
            G = {n: c[n][np.ix_(jy, jx)] for n in ("ux", "uy", "uz")}; G["src"] = c["geom_src"]
    D, K = np.stack(D), np.stack(K)
    # deramp: robustly subtract a plane a + b·x + c·y (bowls are small, so they barely enter the fit)
    if DERAMP:
        XX, YY = np.meshgrid((xs - xs.mean()) / 1e3, (ys - ys.mean()) / 1e3)
        for i in range(D.shape[0]):
            m = np.isfinite(D[i])
            if m.sum() < 1000: continue
            A = np.c_[np.ones(m.sum()), XX[m], YY[m]]; b = D[i][m]
            keep = np.ones(len(b), bool)
            for _ in range(3):
                coef, *_ = np.linalg.lstsq(A[keep], b[keep], rcond=None)
                r = b - A @ coef
                mad = 1.4826 * np.median(np.abs(r[keep] - np.median(r[keep]))) + 1e-9
                keep = np.abs(r - np.median(r[keep])) < 3 * mad
            D[i] = D[i] - (coef[0] + coef[1] * XX + coef[2] * YY)
    # long-wavelength noise filter: subtract from each pair a smoothed (sigma = HP_SIGMA_KM) surface,
    # estimated WITHOUT the areas that clearly move (bowls are masked, so they do not shrink)
    if HP_SIGMA_KM:
        dt0 = np.array([(c["sec"] - c["ref"]).days for c in cl], float)
        sig = HP_SIGMA_KM * 1000 / 80
        def _smooth(a, w):
            num = ndimage.gaussian_filter(np.where(w, a, 0).astype("f8"), sig, mode="nearest")
            den = ndimage.gaussian_filter(w.astype("f8"), sig, mode="nearest")
            return np.where(den > 0.05, num / np.maximum(den, 1e-6), 0), den
        v0 = np.nanmean(D / dt0[:, None, None], 0) * 365.25 * 100
        v0 = np.where(np.isfinite(v0), ndimage.median_filter(np.nan_to_num(v0), size=5), np.nan)  # remove pixel noise
        # "moving" pixels = those that stand out RELATIVE to a smooth background (bowls are local, noise is wide)
        moving = np.zeros(v0.shape, bool)
        for _ in range(2):
            fon, _d = _smooth(v0, np.isfinite(v0) & ~moving)
            moving = ndimage.binary_dilation(np.abs(np.nan_to_num(v0 - fon)) > HP_MASK_CM,
                                             iterations=max(int(500 / 80), 1))    # + 0.5 km margin
        for i in range(D.shape[0]):
            trend, _d = _smooth(D[i], np.isfinite(D[i]) & ~moving)
            D[i] = D[i] - trend
        print(f"  noise filter: {moving.mean():.1%} of pixels masked as moving")
    # reference each pair to a stable area (removes the arbitrary unwrapping offset)
    if REF_LONLAT:
        tr = Transformer.from_crs(4326, cl[0]["epsg"], always_xy=True)
        rx, ry = tr.transform(*REF_LONLAT)
        cx, cy = np.argmin(abs(xs - rx)), np.argmin(abs(ys - ry))
        r = REF_RADIUS_PX
        ref = np.nanmedian(D[:, max(cy-r,0):cy+r+1, max(cx-r,0):cx+r+1], axis=(1, 2))
    else:
        ref = np.nanmedian(D, axis=(1, 2))
    D = D - ref[:, None, None]
    ref_t = np.array([np.datetime64(c["ref"]) for c in cl])
    sec_t = np.array([np.datetime64(c["sec"]) for c in cl])
    dt_days = (sec_t - ref_t).astype("timedelta64[D]").astype(float)
    ds = xr.Dataset(
        dict(disp=(("pair", "y", "x"), D.astype("float32")),
             coh=(("pair", "y", "x"), K.astype("float32"))),
        coords=dict(pair=np.arange(len(cl)), y=ys, x=xs,
                    ref_date=("pair", ref_t), sec_date=("pair", sec_t), dt_days=("pair", dt_days)),
        attrs=dict(epsg=cl[0]["epsg"], wavelength_m=cl[0]["wavelength"], sign=SIGN, units="m LOS",
                   deramp=int(DERAMP), edge_erode_px=EDGE_ERODE_PX, hp_sigma_km=HP_SIGMA_KM))
    if G is not None:
        for n in ("ux", "uy", "uz"):
            ds[f"los_{n[1]}"] = (("y", "x"), G[n])
        ds.attrs["geom_src"] = G["src"]
    # chain breaks (missing 12-day pairs)
    gaps = [(str(sec_t[i])[:10], str(ref_t[i+1])[:10]) for i in range(len(cl)-1) if ref_t[i+1] > sec_t[i]]
    ds.attrs["gaps"] = str(gaps)
    return ds

def velocity(ds):
    rate = ds.disp / ds.dt_days * 365.25 * 100                        # cm/yr for each pair
    valid = np.isfinite(ds.disp).mean("pair")
    v = rate.mean("pair", skipna=True).where(valid >= MIN_VALID_FRAC)
    n = ds.sizes["pair"]; h = max(n // 2, 1)
    acc = (rate.isel(pair=slice(h, None)).mean("pair") - rate.isel(pair=slice(0, h)).mean("pair"))
    cum = (ds.disp.fillna(0).cumsum("pair") * 100).where(valid >= MIN_VALID_FRAC)   # cm
    return v, rate.std("pair"), acc.where(valid >= MIN_VALID_FRAC), valid, cum

stacks, results = {}, {}
for k, cl in crops.items():
    if len(cl) < 3:
        print(k, ": too few pairs (", len(cl), ")"); continue
    try:
        ds = build_stack(cl)
    except ValueError as e:
        print(k, ": skipping —", e); continue
    v, s, a, valid, cum = velocity(ds)
    vv = v.values[np.isfinite(v.values)]
    if vv.size < 100:
        print(k, ": too few valid pixels (", vv.size, ")"); continue
    stacks[k] = ds
    results[k] = dict(v=v, std=s, acc=a, valid=valid, cum=cum)
    ds.to_netcdf(OUT / f"stack_{k}.nc")
    print(f"{k}: {ds.sizes['pair']} pairs, grid {ds.sizes['y']}×{ds.sizes['x']}, valid pixels {vv.size}, "
          f"velocity: 1st percentile {np.percentile(vv, 1):.1f}, min {vv.min():.1f} cm/yr, "
          f"pixels < −50 cm/yr: {(vv < -50).sum()}, breaks: {ds.attrs['gaps']}")

## 6. Isolation Forest: which patches behave unlike the rest

In [ ]:
from sklearn.ensemble import IsolationForest
from scipy import ndimage

def anomalies(k):
    r, ds = results[k], stacks[k]
    v = r["v"].values
    gy, gx = np.gradient(np.nan_to_num(v), 0.08, 0.08)          # cm/yr per km (80 m pixel)
    grad = np.hypot(gx, gy)
    F = np.stack([v, r["std"].values, r["acc"].values, grad,
                  ds.coh.mean("pair").values, r["valid"].values], -1)
    m = np.all(np.isfinite(F), -1)
    X = F[m]; X = (X - X.mean(0)) / (X.std(0) + 1e-9)
    iso = IsolationForest(n_estimators=300, contamination=CONTAMINATION, random_state=0).fit(X)
    score = np.full(v.shape, np.nan, "float32")
    s = -iso.score_samples(X); score[m] = (s - s.min()) / (s.max() - s.min() + 1e-9)
    flag = np.zeros(v.shape, bool); flag[m] = iso.predict(X) == -1
    # "bowl" = an anomaly that SINKS; neighbouring pixels are grouped into zones
    sink = flag & (np.nan_to_num(v) < 0)
    sink = ndimage.binary_opening(sink, iterations=1)
    lab, nz = ndimage.label(sink)
    tr = Transformer.from_crs(ds.attrs["epsg"], 4326, always_xy=True)
    zones = []
    for z in range(1, nz + 1):
        mm = lab == z
        if mm.sum() < MIN_ZONE_PX: continue
        yy, xx = np.nonzero(mm)
        lon, lat = tr.transform(float(ds.x[int(xx.mean())]), float(ds.y[int(yy.mean())]))
        zones.append(dict(zone=z, lon=round(lon, 5), lat=round(lat, 5),
                          area_km2=round(mm.sum() * 0.0064, 2),
                          v_mean_cm_yr=round(float(np.nanmean(v[mm])), 1),
                          v_min_cm_yr=round(float(np.nanmin(v[mm])), 1),
                          acc_cm_yr=round(float(np.nanmean(r["acc"].values[mm])), 1),
                          _iy=int(yy[np.nanargmin(v[mm])]), _ix=int(xx[np.nanargmin(v[mm])])))
    zones.sort(key=lambda z: z["v_min_cm_yr"])
    return score, flag, lab, zones

import json, pandas as pd
anom = {}
for k in results:
    score, flag, lab, zones = anomalies(k)
    anom[k] = dict(score=score, flag=flag, zones=zones)
    df = pd.DataFrame([{kk: vv for kk, vv in z.items() if not kk.startswith("_")} for z in zones])
    df.to_csv(OUT / f"zones_{k}.csv", index=False)
    gj = {"type": "FeatureCollection", "features": [
        {"type": "Feature", "geometry": {"type": "Point", "coordinates": [z["lon"], z["lat"]]},
         "properties": {kk: vv for kk, vv in z.items() if not kk.startswith("_")}} for z in zones]}
    (OUT / f"zones_{k}.geojson").write_text(json.dumps(gj, ensure_ascii=False))
    print(f"\n{k}: {len(zones)} active sinking zones")
    try: display(df.head(10))
    except NameError: print(df.head(10))

## 7. GeoTIFFs and maps

In [ ]:
import rioxarray  # noqa
import matplotlib.pyplot as plt

def to_tif(arr, ds, path):
    da = xr.DataArray(arr, dims=("y", "x"), coords=dict(y=ds.y, x=ds.x)).astype("float32")
    da.rio.write_crs(ds.attrs["epsg"], inplace=True).rio.to_raster(path)

for k, r in results.items():
    ds, a = stacks[k], anom[k]
    to_tif(r["v"].values, ds, OUT / f"velocity_{k}.tif")
    to_tif(a["score"], ds, OUT / f"anomaly_{k}.tif")
    ext = [float(ds.x.min()), float(ds.x.max()), float(ds.y.min()), float(ds.y.max())]
    lim = np.nanpercentile(np.abs(r["v"].values), 99)
    fig, ax = plt.subplots(1, 2, figsize=(15, 6))
    im = ax[0].imshow(r["v"], extent=ext, cmap="RdBu", vmin=-lim, vmax=lim)
    plt.colorbar(im, ax=ax[0], label="cm/yr (LOS, negative = away from the satellite)")
    ax[0].set_title(f"Ostrava–Karviná · velocity · track {k}")
    im = ax[1].imshow(a["score"], extent=ext, cmap="magma")
    plt.colorbar(im, ax=ax[1], label="anomaly score (Isolation Forest)")
    for i, z in enumerate(a["zones"][:8], 1):
        px, py = float(ds.x[z["_ix"]]), float(ds.y[z["_iy"]])
        for axx in ax: axx.annotate(str(i), (px, py), color="lime", fontsize=11, weight="bold")
    ax[1].set_title("Anomalous patches · numbers = top bowls")
    for axx in ax: axx.set_xlabel(f"EPSG:{ds.attrs['epsg']}  x, m")
    plt.tight_layout(); plt.savefig(OUT / f"map_{k}.png", dpi=150); plt.show()

    # time series of the three most active bowls
    t = pd.to_datetime(ds.sec_date.values)
    fig, ax = plt.subplots(figsize=(10, 4))
    for i, z in enumerate(a["zones"][:3], 1):
        ax.plot(t, r["cum"][:, z["_iy"], z["_ix"]], "o-", label=f"bowl {i}: {z['lat']:.3f}N {z['lon']:.3f}E")
    ax.axhline(0, color="gray", lw=.5); ax.set_ylabel("cumulative LOS displacement, cm")
    ax.set_title(f"How the ground sinks over the mines · track {k}"); ax.legend()
    plt.tight_layout(); plt.savefig(OUT / f"timeseries_{k}.png", dpi=150); plt.show()
print(sorted(p.name for p in OUT.iterdir()))

## 8. Vertical and east–west from two tracks

Each track sees motion only along its own line of sight. The ascending track (A) looks east, the descending (D) west, so
for each pixel we solve `v_A = e_A·E + u_A·U`, `v_D = e_D·E + u_D·U`.
The north component is dropped: both tracks fly almost along the meridian, so it is nearly invisible.
Result: **U** (up/down, cm/yr) and **E** (east/west).

In [ ]:
def _da(arr, ds):
    da = xr.DataArray(np.asarray(arr, "f4"), dims=("y", "x"), coords=dict(y=ds.y, x=ds.x))
    return da.rio.write_crs(ds.attrs["epsg"]).rio.write_nodata(np.nan)

def _best(direction):
    c = [k for k in results if k.startswith(direction) and "los_x" in stacks[k]]
    return max(c, key=lambda k: int(np.isfinite(results[k]["v"].values).sum())) if c else None

kA, kD = DECOMP_PAIR if DECOMP_PAIR else (_best("A"), _best("D"))
if not (kA and kD):
    print("Need an ascending and a descending track with geometry. Have:", list(results))
else:
    dA, dD = stacks[kA], stacks[kD]
    print(f"Pair: {kA} (geometry: {dA.attrs.get('geom_src')})  +  {kD} (geometry: {dD.attrs.get('geom_src')})")
    vA = _da(results[kA]["v"].values, dA)
    eA, uA = _da(dA.los_x, dA), _da(dA.los_z, dA)
    from rasterio.enums import Resampling
    rm = lambda arr: _da(arr, dD).rio.reproject_match(vA, resampling=Resampling.bilinear)
    vD, eD, uD = rm(results[kD]["v"].values), rm(dD.los_x), rm(dD.los_z)
    det = eA * uD - uA * eD
    ok = np.abs(det) > 0.2
    U = ((eA * vD - vA * eD) / det).where(ok)
    E = ((vA * uD - uA * vD) / det).where(ok)
    U.rio.to_raster(OUT / "vertical_cm_yr.tif"); E.rio.to_raster(OUT / "east_cm_yr.tif")
    uu = U.values[np.isfinite(U.values)]
    print(f"Vertical: {uu.size} pixels, 1st percentile {np.percentile(uu, 1):.1f} cm/yr, "
          f"median {np.median(uu):.2f} cm/yr (should be ≈0)")

    # ---------- OSM: water and spoil heaps (to drop artefacts and explain the bowls) ----------
    from rasterio import features
    import geopandas as gpd
    def _osm_layer(tags, cache, override):
        if override: return gpd.read_file(override)
        f = OUT / cache
        if f.exists(): return gpd.read_file(f)
        try:
            import osmnx as ox
            ox.settings.http_user_agent = "space-apps-research"   # the team's User-Agent for outside services
            try:
                g = ox.features_from_bbox(bbox=BBOX, tags=tags)                       # osmnx >= 2.0
            except TypeError:
                g = ox.features_from_bbox(BBOX[3], BBOX[1], BBOX[2], BBOX[0], tags=tags)  # osmnx 1.x
            g = g[g.geometry.geom_type.isin(["Polygon", "MultiPolygon"])][["geometry"]].reset_index(drop=True)
            g.to_file(f, driver="GeoJSON")
            return g
        except Exception as e:
            print(f"  OSM ({cache}) unavailable: {e!r:.120}")
            return None
    def _rast(g):
        if g is None or len(g) == 0: return np.zeros(U.shape, bool)
        g = g.to_crs(U.rio.crs)
        return features.rasterize(((geom, 1) for geom in g.geometry), out_shape=U.shape,
                                  transform=U.rio.transform(), fill=0, all_touched=True).astype(bool)
    water = _rast(_osm_layer({"natural": "water", "landuse": ["reservoir", "basin"], "waterway": "riverbank"},
                             "osm_water.geojson", WATER_FILE))
    heap = _rast(_osm_layer({"man_made": "spoil_heap", "landuse": "landfill"}, "osm_heaps.geojson", HEAP_FILE))
    print(f"  water: {water.mean():.1%} of the box, spoil heaps: {heap.mean():.1%} of the box")

    # ---------- BOWL CLUSTERS ----------
    # 1) subsidence mask after light smoothing (water cannot be "subsidence")
    Us = ndimage.median_filter(np.nan_to_num(U.values), size=3)
    mask = ndimage.binary_opening(Us < VERT_ZONE_CM) & ~water
    # 1b) white centres: small no-data areas (< MAX_HOLE_KM2) next to subsidence are part of the bowl too
    nodata = ~np.isfinite(U.values) & ~water          # a reservoir is not a "white centre"
    nl, nn = ndimage.label(nodata)
    near = ndimage.binary_dilation(mask, iterations=2)
    sizes = ndimage.sum(nodata, nl, index=np.arange(1, nn + 1)) * 0.0064
    touch = ndimage.maximum(near, nl, index=np.arange(1, nn + 1))
    holes = np.isin(nl, 1 + np.nonzero((sizes < MAX_HOLE_KM2) & (touch > 0))[0])
    mask = mask | holes
    # 2) merge fragments closer than CLUSTER_GAP_M and fill the white centres (the radar lost the ground there)
    r = max(int(round(CLUSTER_GAP_M / 2 / 80)), 1)
    disk = np.hypot(*np.mgrid[-r:r+1, -r:r+1]) <= r
    merged = ndimage.binary_fill_holes(ndimage.binary_closing(mask, structure=disk, iterations=1))
    base, nbase = ndimage.label(merged)
    # 2b) split bowls with several centres (watershed): a centre is a local minimum separated by a "pass" >= SPLIT_DEPTH_CM
    from skimage.segmentation import watershed
    from skimage.morphology import h_maxima
    Uf = U.values.copy()
    nanm = ~np.isfinite(Uf)
    if nanm.any():                                   # fill white centres with the nearest values
        idx = ndimage.distance_transform_edt(nanm, return_distances=False, return_indices=True)
        Uf = Uf[tuple(idx)]
    Uf = ndimage.gaussian_filter(Uf, SPLIT_SMOOTH_PX)
    mk = h_maxima(-Uf, SPLIT_DEPTH_CM).astype(bool) & merged & (Uf < VERT_ZONE_CM)
    markers, nmk = ndimage.label(mk)
    for z in range(1, nbase + 1):                    # a cluster without a clear centre gets one marker at its deepest point
        m = base == z
        if not markers[m].any():
            iy, ix = np.unravel_index(np.argmin(np.where(m, Uf, np.inf)), Uf.shape)
            nmk += 1; markers[iy, ix] = nmk
    lab = watershed(Uf, markers, mask=merged)
    n_split = sum(len(np.unique(lab[base == z])) > 1 for z in range(1, nbase + 1))
    print(f"  clusters before the split: {nbase}, after: {len(np.unique(lab)) - 1} (split {n_split})")
    # 3) LOS velocities of all tracks on the vertical grid (for confirmation)
    los_on_grid = {k: _da(results[k]["v"].values, stacks[k]).rio.reproject_match(U, resampling=Resampling.bilinear).values
                   for k in results}
    tr = Transformer.from_crs(dA.attrs["epsg"], 4326, always_xy=True)
    trb = Transformer.from_crs(4326, dA.attrs["epsg"], always_xy=True)
    Xg, Yg = np.meshgrid(U.x.values, U.y.values)
    clusters, rejected, rejected_water = [], 0, 0
    for z in np.unique(lab)[1:]:
        mm = lab == z
        if mm.sum() < MIN_ZONE_PX: continue
        water_frac = float((mm & water).sum() / mm.sum())
        if water_frac > MAX_WATER_FRAC:
            rejected_water += 1; continue
        valid = mm & np.isfinite(U.values) & ~water
        if valid.sum() < MIN_ZONE_PX // 2: continue
        heap_frac = float((mm & heap).sum() / mm.sum())
        on_h, off_h = valid & heap, valid & ~heap
        conf = {k: float(np.nanmean(v[mm])) for k, v in los_on_grid.items() if np.isfinite(v[mm]).sum() >= 5}
        n_conf = sum(val < LOS_CONFIRM_CM for val in conf.values())
        if n_conf < min(MIN_CONFIRM_TRACKS, len(los_on_grid)):
            rejected += 1; continue
        cy, cx = ndimage.center_of_mass(mm)
        cxm, cym = float(np.interp(cx, np.arange(len(U.x)), U.x.values)), float(np.interp(cy, np.arange(len(U.y)), U.y.values))
        lon, lat = tr.transform(cxm, cym)
        # horizontal check: west of the centre motion should be eastward (+), east of the centre westward (−)
        ring = ndimage.binary_dilation(mm, iterations=max(int(500 / 80), 1)) & np.isfinite(E.values)
        e_w = float(np.nanmean(E.values[ring & (Xg < cxm)])) if (ring & (Xg < cxm)).any() else np.nan
        e_e = float(np.nanmean(E.values[ring & (Xg > cxm)])) if (ring & (Xg > cxm)).any() else np.nan
        conv = e_w - e_e
        clusters.append(dict(
            lon=round(lon, 5), lat=round(lat, 5),
            area_km2=round(mm.sum() * 0.0064, 2),
            no_data_frac=round(1 - valid.sum() / mm.sum(), 2),        # share of the "white centre"
            up_mean_cm_yr=round(float(np.nanmean(U.values[valid])), 1),
            up_p10_cm_yr=round(float(np.nanpercentile(U.values[valid], 10)), 1),
            east_west_side=round(e_w, 1), east_east_side=round(e_e, 1),
            horiz_to_center=("yes" if conv > 0 else "no") if np.isfinite(conv) else "?",
            n_tracks_confirm=n_conf,
            group=int(np.bincount(base[mm]).argmax()),            # shared number for the parts of one bowl
            water_frac=round(water_frac, 2), heap_frac=round(heap_frac, 2),
            up_mean_on_heap=round(float(np.nanmean(U.values[on_h])), 1) if on_h.sum() >= 5 else None,
            up_mean_off_heap=round(float(np.nanmean(U.values[off_h])), 1) if off_h.sum() >= 5 else None,
            **{f"los_{k}": round(v, 1) for k, v in conf.items()},
            _z=z))
    clusters.sort(key=lambda c: c["up_mean_cm_yr"])
    for i, c in enumerate(clusters, 1):
        c["id"] = f"M{i:02d}"; c["name"] = ""   # names are added later from OSM (nearest town / mine)
    print(f"Bowls: {len(clusters)} (rejected as unconfirmed: {rejected}, on water: {rejected_water})")

    # bowl polygons -> GeoJSON (lon/lat)
    from shapely.geometry import shape, mapping
    from shapely.ops import transform as shp_transform, unary_union
    feats = []
    for c in clusters:
        m = (lab == c["_z"]).astype("uint8")
        geoms = [shape(g) for g, v in features.shapes(m, mask=m.astype(bool), transform=U.rio.transform()) if v == 1]
        poly = unary_union(geoms)
        poly = shp_transform(lambda x, y, z=None: tr.transform(x, y), poly)  # noqa
        feats.append({"type": "Feature", "geometry": mapping(poly),
                      "properties": {k: v for k, v in c.items() if not k.startswith("_")}})
    (OUT / "zones_vertical_clustered.geojson").write_text(json.dumps({"type": "FeatureCollection", "features": feats}, ensure_ascii=False))
    dfc = pd.DataFrame([{k: v for k, v in c.items() if not k.startswith("_")} for c in clusters])
    dfc.to_csv(OUT / "zones_vertical_clustered.csv", index=False)
    cols = ["id", "group", "lon", "lat", "area_km2", "no_data_frac", "water_frac", "heap_frac", "up_mean_cm_yr",
            "up_p10_cm_yr", "up_mean_on_heap", "up_mean_off_heap", "horiz_to_center", "n_tracks_confirm"]
    try: display(dfc[cols])
    except NameError: print(dfc[cols].to_string())
    ok_h = (dfc["horiz_to_center"] == "yes").mean() if len(dfc) else np.nan
    print(f"Horizontal motion points to the centre in {ok_h:.0%} of bowls. Below 50 % the east sign in the geometry is probably flipped.")

    ext = [float(U.x.min()), float(U.x.max()), float(U.y.min()), float(U.y.max())]
    fig, ax = plt.subplots(1, 2, figsize=(15, 6))
    lim = max(np.nanpercentile(np.abs(U.values), 99), 5)
    im = ax[0].imshow(U, extent=ext, cmap="RdBu", vmin=-lim, vmax=lim)
    plt.colorbar(im, ax=ax[0], label="vertical, cm/yr (negative = subsidence)")
    ax[0].set_title(f"Ground down/up · {kA} + {kD}")
    for axx in ax:
        axx.contour(U.x, U.y, np.isin(lab, [c["_z"] for c in clusters]), levels=[0.5], colors="lime", linewidths=1)
    for c in clusters[:12]:
        px, py = trb.transform(c["lon"], c["lat"])
        ax[0].annotate(c["id"], (px, py), color="black", fontsize=9, weight="bold",
                       bbox=dict(boxstyle="round,pad=0.1", fc="lime", alpha=.8))
    lim = max(np.nanpercentile(np.abs(E.values), 99), 5)
    im = ax[1].imshow(E, extent=ext, cmap="PuOr", vmin=-lim, vmax=lim)
    plt.colorbar(im, ax=ax[1], label="east (+) / west (−), cm/yr")
    ax[1].set_title("Horizontal motion (east–west) · bowl outlines")
    plt.tight_layout(); plt.savefig(OUT / "map_vertical_east.png", dpi=150); plt.show()

## 9. Checks and next steps

**Quality control (required before anything goes on a plate):**
- Bowls must be **negative**. If they come out positive, set `SIGN = +1`.
- The ascending (A) and descending (D) tracks must show the same bowls in the same places. A zone seen on one track only
  is most likely noise or an atmospheric artefact.
- Compare with **EGMS** (https://egms.land.copernicus.eu, product *Ortho – Vertical*, free after registration): Sentinel-1
  since 2018; the Karviná bowls should match in place. Absolute values will differ (LOS vs vertical, different years).
- In the centres of active bowls coherence often drops because the ground moves too fast. These holes are part of the story.
- The scene median of the vertical should be ≈0. If it is clearly shifted, set `REF_LONLAT` at a stable place.
- Horizontal motion over a bowl should point to its centre (eastward on the west side, westward on the east side). If the
  picture is mirrored, check the geometry (the "geometry: …" line printed in section 4).

**Next:** run `scripts/ostrava/ostrava_osm.mjs`, `ostrava_map.mjs` and `build_ostrava_results.mjs` (see `README.md`);
compare the bowls with EGMS and with the mining-lease maps (PL: MIDAS PIG-PIB, CZ: ČGS geoportal).